<a href="https://colab.research.google.com/github/miriamamin1213-ux/Seed42_Models/blob/main/CNN42_Hancock.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [7]:
# ============================================================
# CNN42_Hancock
# Same CNN Hyperparameters as CNN42_Hecktor
# ============================================================

from google.colab import drive
drive.mount('/content/drive')

import os
import json
import random
import numpy as np
import pandas as pd

import torch
import torch.nn as nn

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import (
    LabelEncoder,
    StandardScaler
)

from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    balanced_accuracy_score,
    roc_auc_score,
    f1_score
)

# ------------------------------------------------------------
# Reproducibility
# ------------------------------------------------------------

def seed_everything(seed=42):

    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    np.random.seed(seed)
    random.seed(seed)

    os.environ["PYTHONHASHSEED"] = str(seed)
    os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"

    torch.use_deterministic_algorithms(
        True,
        warn_only=True
    )

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

    torch.backends.cuda.matmul.allow_tf32 = False
    torch.backends.cudnn.allow_tf32 = False

SEED = 42

seed_everything(SEED)

# ------------------------------------------------------------
# Read Data
# ------------------------------------------------------------

with open('/content/drive/MyDrive/clinical_data.json') as f:
    clinical = pd.DataFrame(json.load(f))

with open('/content/drive/MyDrive/pathological_data.json') as f:
    pathology = pd.DataFrame(json.load(f))

df = clinical.merge(
    pathology,
    on='patient_id',
    how='inner'
)

df = df[
    df['hpv_association_p16'].isin(
        ['positive', 'negative']
    )
].copy()

df['HPV'] = df['hpv_association_p16'].map({
    'negative': 0,
    'positive': 1
})

# ------------------------------------------------------------
# Build Modelling Dataset
# ------------------------------------------------------------

df_model = df[
[
    'age_at_initial_diagnosis',
    'sex',
    'smoking_status',
    'primarily_metastasis',
    'first_treatment_intent',
    'first_treatment_modality',
    'days_to_first_treatment',
    'adjuvant_treatment_intent',
    'adjuvant_radiotherapy',
    'adjuvant_radiotherapy_modality',
    'adjuvant_systemic_therapy',
    'adjuvant_systemic_therapy_modality',
    'adjuvant_radiochemotherapy',
    'primary_tumor_site',
    'pT_stage',
    'pN_stage',
    'histologic_type',
    'number_of_positive_lymph_nodes',
    'number_of_resected_lymph_nodes',
    'perinodal_invasion',
    'lymphovascular_invasion_L',
    'vascular_invasion_V',
    'perineural_invasion_Pn',
    'resection_status',
    'infiltration_depth_in_mm',
    'HPV'
]
].copy()

# ------------------------------------------------------------
# Missing Values
# ------------------------------------------------------------

cat_cols = [
    'sex',
    'smoking_status',
    'primarily_metastasis',
    'first_treatment_intent',
    'first_treatment_modality',
    'adjuvant_treatment_intent',
    'adjuvant_radiotherapy',
    'adjuvant_radiotherapy_modality',
    'adjuvant_systemic_therapy',
    'adjuvant_systemic_therapy_modality',
    'adjuvant_radiochemotherapy',
    'primary_tumor_site',
    'pT_stage',
    'pN_stage',
    'histologic_type',
    'perinodal_invasion',
    'lymphovascular_invasion_L',
    'vascular_invasion_V',
    'perineural_invasion_Pn',
    'resection_status'
]

for col in cat_cols:
    df_model[col] = df_model[col].fillna(
        df_model[col].mode()[0]
    )

num_cols = [
    'age_at_initial_diagnosis',
    'days_to_first_treatment',
    'number_of_positive_lymph_nodes',
    'number_of_resected_lymph_nodes',
    'infiltration_depth_in_mm'
]

for col in num_cols:
    df_model[col] = df_model[col].fillna(
        df_model[col].median()
    )

# ------------------------------------------------------------
# Label Encoding
# ------------------------------------------------------------

for col in cat_cols:
    df_model[col] = LabelEncoder().fit_transform(
        df_model[col].astype(str)
    )

# ------------------------------------------------------------
# Features and Target
# ------------------------------------------------------------

X = df_model.drop(columns=['HPV'])
y = df_model['HPV']

# ------------------------------------------------------------
# Train/Test Split
# ------------------------------------------------------------

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=SEED,
    stratify=y
)

# ------------------------------------------------------------
# Standardisation
# ------------------------------------------------------------

scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# ------------------------------------------------------------
# Torch Conversion
# ------------------------------------------------------------

X_train = torch.FloatTensor(X_train)
X_test = torch.FloatTensor(X_test)

y_train = torch.LongTensor(y_train.values)
y_test = torch.LongTensor(y_test.values)

X_train = X_train.unsqueeze(1)
X_test = X_test.unsqueeze(1)

n_features = X.shape[1]

# ------------------------------------------------------------
# MODEL
# Conv16-FC32-FC16-FC2
# ------------------------------------------------------------

class HPVCNN(nn.Module):

    def __init__(self):

        super().__init__()

        self.conv1 = nn.Conv1d(
            in_channels=1,
            out_channels=16,
            kernel_size=3,
            padding=1
        )

        self.relu = nn.ReLU()

        self.flatten = nn.Flatten()

        self.fc1 = nn.Linear(
            16 * n_features,
            32
        )

        self.fc2 = nn.Linear(
            32,
            16
        )

        self.fc3 = nn.Linear(
            16,
            2
        )

    def forward(self, x):

        x = self.conv1(x)

        x = self.relu(x)

        x = self.flatten(x)

        x = self.relu(self.fc1(x))
        x = self.relu(self.fc2(x))
        x = self.fc3(x)

        return x

# ------------------------------------------------------------
# Model Setup
# ------------------------------------------------------------

model = HPVCNN()

weights = torch.tensor(
    [2.0, 1.0],
    dtype=torch.float32
)

criterion = nn.CrossEntropyLoss(
    weight=weights
)

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.0001
)

# ------------------------------------------------------------
# Training
# ------------------------------------------------------------

epochs = 500

best_test_loss = float("inf")
best_epoch = 0

for epoch in range(epochs):

    model.train()

    outputs = model(X_train)

    train_loss = criterion(
        outputs,
        y_train
    )

    optimizer.zero_grad()

    train_loss.backward()

    optimizer.step()

    model.eval()

    with torch.no_grad():

        test_outputs = model(X_test)

        test_loss = criterion(
            test_outputs,
            y_test
        )

    if test_loss.item() < best_test_loss:

        best_test_loss = test_loss.item()

        best_epoch = epoch + 1

        torch.save(
            model.state_dict(),
            "best_cnn_hancock.pth"
        )

    if (epoch + 1) % 50 == 0:

        print(
            f"Epoch {epoch+1}, "
            f"Train={train_loss.item():.4f}, "
            f"Test={test_loss.item():.4f}"
        )

print()
print("Best Test Loss =", round(best_test_loss, 4))
print("Best Epoch =", best_epoch)

# ------------------------------------------------------------
# Load Best Model
# ------------------------------------------------------------

model.load_state_dict(
    torch.load("best_cnn_hancock.pth")
)

model.eval()

with torch.no_grad():

    outputs = model(X_test)

    probabilities = torch.softmax(
        outputs,
        dim=1
    )

    predicted = torch.argmax(
        outputs,
        dim=1
    )

# ------------------------------------------------------------
# Results
# ------------------------------------------------------------

print()
print("Classification Report\n")

print(
    classification_report(
        y_test.numpy(),
        predicted.numpy(),
        digits=4
    )
)

# ------------------------------------------------------------
# Confusion Matrix
# ------------------------------------------------------------

cm = confusion_matrix(
    y_test.numpy(),
    predicted.numpy()
)

print("Confusion Matrix")
print(cm)

# ------------------------------------------------------------
# Metrics
# ------------------------------------------------------------

bal_acc = balanced_accuracy_score(
    y_test.numpy(),
    predicted.numpy()
)

f1 = f1_score(
    y_test.numpy(),
    predicted.numpy()
)

auc = roc_auc_score(
    y_test.numpy(),
    probabilities[:,1].numpy()
)

accuracy = (
    predicted.eq(y_test)
    .sum()
    .item()
    /
    len(y_test)
)

print()
print(f"Accuracy:            {accuracy:.4f}")
print(f"Balanced Accuracy:   {bal_acc:.4f}")
print(f"F1-score:            {f1:.4f}")
print(f"AUC:                 {auc:.4f}")

# ------------------------------------------------------------
# Model Summary
# ------------------------------------------------------------

print()
print("====================================")
print("CNN42_HANCOCK SUMMARY")
print("====================================")
print("Architecture : Conv16-FC32-FC16-2")
print("Input Features :", n_features)
print("Learning Rate : 0.0001")
print("Optimiser : Adam")
print("Class Weights : [2.0,1.0]")
print("SMOTE : No")
print("Epochs : 500")
print("Best Epoch :", best_epoch)
print("Best Test Loss :", round(best_test_loss,4))
print("Training Patients :", len(y_train))
print("Test Patients :", len(y_test))
print("Seed :", SEED)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Epoch 50, Train=0.6130, Test=0.6027
Epoch 100, Train=0.5586, Test=0.5459
Epoch 150, Train=0.5292, Test=0.5167
Epoch 200, Train=0.4992, Test=0.4880
Epoch 250, Train=0.4682, Test=0.4644
Epoch 300, Train=0.4419, Test=0.4493
Epoch 350, Train=0.4170, Test=0.4399
Epoch 400, Train=0.3925, Test=0.4317
Epoch 450, Train=0.3679, Test=0.4238
Epoch 500, Train=0.3426, Test=0.4158

Best Test Loss = 0.4158
Best Epoch = 500

Classification Report

              precision    recall  f1-score   support

           0     0.7200    0.9231    0.8090        39
           1     0.8235    0.5000    0.6222        28

    accuracy                         0.7463        67
   macro avg     0.7718    0.7115    0.7156        67
weighted avg     0.7633    0.7463    0.7309        67

Confusion Matrix
[[36  3]
 [14 14]]

Accuracy:            0.7463
Balanced Accuracy:   0.7115
F1-score:       